# Spoke model

`DriftFluid2D` is a drift fluid in the plane of a magnetron discharge: `x` from the
cathode to the anode across the magnetic field, `y` azimuthal and periodic. Electrons drift
in $E \times B$ and diffuse, ions are unmagnetised, Poisson's equation closes the
potential. Azimuthal structures (spokes) grow from it; `spoke_report` describes a run in the
terms of the experiments. Here the smallest check: in a uniform plasma, periodic in `x`
and `y`, a small azimuthal perturbation grows at the rate of the local dispersion
relation. The rates are synthetic.

In [ ]:
import numpy as np
import plasma_plots
from plasmacoll.constants import ELEMENTARY_CHARGE as e

from wellbalanced import DriftFluid2D, spoke_report
from wellbalanced.driftfluid2d import uniform_plasma_roots

density, e_field, b_field, te = 1e16, -2200.0, 0.02, 5.0  # m^-3, V/m, T, eV
gas, nu_e, nu_i = 1.43e20, 2e7, 1e6  # m^-3, 1/s, 1/s
plasma = DriftFluid2D(
    0.0048,  # x: 4.8 mm
    0.016,  # y: 16 mm, the azimuthal period
    4,
    32,
    voltage=0.0,
    magnetic_field=b_field,
    gas_density=gas,
    electron_temperature=te,
    momentum_rate=nu_e / gas,
    ion_friction_rate=nu_i / gas,
    ionization_rate=lambda t: 0.0 * t,
    refill_rate=0.0,
    secondary_emission=0.0,
    ion_temperature=1e-4,
    field_bc=e_field,  # a uniform axial field instead of the electrodes
    periodic_x=True,
)

The ions start at their drift velocity in the field, the stationary sources hold the
uniform state, and mode `m = 1` gets a perturbation of 1e-4:

In [ ]:
start = plasma.initial_state(density, electron_temperature=te)
start.ion_velocity[0] = e * e_field / (plasma.ion_mass * nu_i)
start = plasma.step(start, 1e-13)
plasma.stationary_sources(start)
k = 2 * np.pi / plasma.length_y
wave = 1.0 + 1e-4 * np.cos(k * plasma.y)[None, :]
start.electron_density = start.electron_density * wave
start.ion_density = start.ion_density * wave

roots = uniform_plasma_roots(
    0.0, k, density, e_field, b_field, te, plasma.ion_mass, nu_i, nu_e
)
theory = roots[np.argmax(roots.real)]
theory  # gamma - i omega (1/s)

In [ ]:
t_end = 3.0 / theory.real
run = plasma.run(start, t_end, record_every=t_end / 120)
run.steps

The run records the electron density along `y` at a probe (mid-gap) and its azimuthal
Fourier amplitudes:

In [ ]:
record = run.to_dataset()
record = record.assign_coords(time=record.time * 1e6, y=record.y * 1e3)
record.time.attrs.update(long_name="t", units="µs")
record.y.attrs.update(long_name="y (azimuthal)", units="mm")
with plasma_plots.figure(1, 2, figsize=(11, 3.8)) as fig:
    record.probe.plasma.plot.slice(x="y", y="time", ax=fig[0])
    record.mode_amplitude.sel(mode=[1, 2]).plasma.plot.profiles(
        x="time", over="mode", logy=True, ax=fig[1]
    )

## The spoke report

`spoke_report` takes the dominant mode over a window (the last half of the run by default)
and gives its growth rate, phase velocity, direction against $E \times B$, and the
modulation and potential hump it makes:

In [ ]:
report = spoke_report(run, (0.45 * t_end, t_end))
report

In [ ]:
print(f"growth rate {report.growth_rate:.3e} /s, theory {theory.real:.3e} /s")
print(
    f"frequency {report.phase_velocity * k:.3e} rad/s, theory {-theory.imag:.3e} rad/s"
)

The mode moves along $E \times B$ (`relative_direction = 1`) at a fraction of the drift
speed. The full magnetron, with electrodes, an exponential field and ionization
(`exponential_field`, the `argon_*_rate` placeholders), uses the same `run` and
`spoke_report`; the [drift fluid verification](/wellbalanced/full-examples/spoke-drift-fluid-2d/)
example has the other checks.